# AI for Email Topic Categorization
This notebook contains code for KNN, Decision Tree, and Linear Regression implemented from scratch.

## 1. Import Required Libraries

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split


ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

## 2. Prepare Email Dataset (30 Emails)

In [3]:

emails = [
    # Spam (15)
    "Congratulations! You've won a $1000 gift card. Click here to claim.",
    "Get rich quick by working from home. Limited offer!",
    "Win a brand new iPhone. Apply now!",
    "You've been selected for a cash reward. Respond ASAP.",
    "Act now! Lowest price ever on our new product.",
    "Cheap medication available online. Order today!",
    "Get your credit score improved in 24 hours.",
    "Claim your free Netflix subscription now.",
    "Special promotion: Buy one, get two free.",
    "Exclusive deal just for you! Don't miss out.",
    "Earn money fast by completing surveys.",
    "Final notice: Your car warranty is expiring.",
    "You have been chosen for a $5000 grant.",
    "Instant approval for your loan application.",
    "Double your income with zero investment.",
    # Ham (15)
    "Can you send me the project report by Monday?",
    "Let's have a meeting at 3PM tomorrow.",
    "Please review the attached document before submission.",
    "Thanks for your help with the client presentation.",
    "I'll be out of office next week, please coordinate with Ali.",
    "Reminder: Parent-teacher meeting on Friday.",
    "The invoice for your last order is attached.",
    "Don't forget to submit your assignment on time.",
    "Are you joining us for lunch today?",
    "Good luck with your exams next week.",
    "Here is the schedule for the upcoming workshop.",
    "Team meeting rescheduled to 11:30 AM.",
    "Check your calendar for the updated invite.",
    "Please update your timesheet by EOD.",
    "Looking forward to catching up this weekend."
]
labels = ['spam'] * 15 + ['ham'] * 15

df = pd.DataFrame({'email': emails, 'label': labels})
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()


NameError: name 'pd' is not defined

## 3. Convert Text to Numbers (Bag of Words)

In [ ]:

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['email']).toarray()
y = df['label_num'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


## 4. K-Nearest Neighbors (KNN) - From Scratch

In [ ]:

def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

def knn_predict(X_train, y_train, x_test, k=3):
    distances = [euclidean_distance(x_test, x) for x in X_train]
    k_indices = np.argsort(distances)[:k]
    k_nearest_labels = [y_train[i] for i in k_indices]
    most_common = Counter(k_nearest_labels).most_common(1)
    return most_common[0][0]

y_pred_knn = [knn_predict(X_train, y_train, x, k=3) for x in X_test]


## 5. Decision Tree - From Scratch

In [ ]:

def gini_index(groups, classes):
    n_instances = float(sum([len(group) for group in groups]))
    gini = 0.0
    for group in groups:
        size = float(len(group))
        if size == 0:
            continue
        score = 0.0
        for class_val in classes:
            proportion = [row[-1] for row in group].count(class_val) / size
            score += proportion * proportion
        gini += (1.0 - score) * (size / n_instances)
    return gini

def test_split(index, value, dataset):
    left, right = [], []
    for row in dataset:
        if row[index] < value:
            left.append(row)
        else:
            right.append(row)
    return left, right

def get_split(dataset):
    class_values = list(set(row[-1] for row in dataset))
    b_index, b_value, b_score, b_groups = 999, 999, 999, None
    for index in range(len(dataset[0]) - 1):
        for row in dataset:
            groups = test_split(index, row[index], dataset)
            gini = gini_index(groups, class_values)
            if gini < b_score:
                b_index, b_value, b_score, b_groups = index, row[index], gini, groups
    return {'index': b_index, 'value': b_value, 'groups': b_groups}

def to_terminal(group):
    outcomes = [row[-1] for row in group]
    return max(set(outcomes), key=outcomes.count)

def split(node, max_depth, min_size, depth):
    left, right = node['groups']
    del(node['groups'])
    if not left or not right:
        node['left'] = node['right'] = to_terminal(left + right)
        return
    if depth >= max_depth:
        node['left'] = to_terminal(left)
        node['right'] = to_terminal(right)
        return
    if len(left) <= min_size:
        node['left'] = to_terminal(left)
    else:
        node['left'] = get_split(left)
        split(node['left'], max_depth, min_size, depth+1)
    if len(right) <= min_size:
        node['right'] = to_terminal(right)
    else:
        node['right'] = get_split(right)
        split(node['right'], max_depth, min_size, depth+1)

def build_tree(train, max_depth, min_size):
    root = get_split(train)
    split(root, max_depth, min_size, 1)
    return root

def predict(node, row):
    if row[node['index']] < node['value']:
        return predict(node['left'], row) if isinstance(node['left'], dict) else node['left']
    else:
        return predict(node['right'], row) if isinstance(node['right'], dict) else node['right']

train_data = [list(X_train[i]) + [y_train[i]] for i in range(len(X_train))]
test_data = [list(X_test[i]) + [y_test[i]] for i in range(len(X_test))]
tree = build_tree(train_data, max_depth=3, min_size=1)
y_pred_tree = [predict(tree, row[:-1]) for row in test_data]


## 6. Evaluation Metrics

In [ ]:

def evaluate_classification(y_true, y_pred):
    tp = sum((y_true[i] == 1 and y_pred[i] == 1) for i in range(len(y_true)))
    tn = sum((y_true[i] == 0 and y_pred[i] == 0) for i in range(len(y_true)))
    fp = sum((y_true[i] == 0 and y_pred[i] == 1) for i in range(len(y_true)))
    fn = sum((y_true[i] == 1 and y_pred[i] == 0) for i in range(len(y_true)))
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return accuracy, precision, recall

evaluate_classification(y_test, y_pred_knn), evaluate_classification(y_test, y_pred_tree)
